In [ ]:
# FONTE : https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar
# CENSO 2023

# BIBLIOTECAS

import pandas as pd
import numpy as np
from google.colab import files
# UPLOAD CSV

print("Realize o upload do arquivo do Censo Escolar (CSV)")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# LEITURA DO CSV
cat_cols = ['TP_DEPENDENCIA', 'TP_LOCALIZACAO', 'SG_UF']
df = pd.read_csv(
    file_name,
    sep=';',
    encoding='latin1',
    low_memory=False,
    dtype={col: 'category' for col in cat_cols}
)
print("Arquivo CSV lido com sucesso")
print("Shape inicial:", df.shape)

# FILTRO ESCOLAS ATIVAS
df = df.query("TP_SITUACAO_FUNCIONAMENTO == 1 & IN_ESCOLARIZACAO == 1")
print("filtro de escolas ativas:", df.shape)

# TRATAMENTO
# Substituição de valores
df.replace({88888: np.nan}, inplace=True)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].replace(9, np.nan)
print("Valores divergentes tratados com sucesso")

# VARIÁVEIS ANALÍTICAS - VETORIZADAS
# Função auxiliar para soma segura de colunas (NaN → 0)
def safe_sum(*cols):
    return sum(df[c].fillna(0) if c in df.columns else 0 for c in cols)

# Índice de tecnologia
df['score_tecnologia'] = safe_sum('IN_INTERNET', 'IN_BANDA_LARGA', 'IN_INTERNET_ALUNOS')

# Computadores por aluno
qt_matriculas = (safe_sum('QT_MAT_FUND_AI', 'QT_MAT_FUND_AF'))
df['computadores_por_aluno'] = (safe_sum('QT_COMP_PORTATIL_ALUNO', 'QT_DESKTOP_ALUNO')) / qt_matriculas

# Relação aluno/professor (fundamental, anos iniciais e finais)
df['aluno_professor_ed_basica_a_iniciais'] = (df['QT_MAT_FUND_AI'] / df["QT_TUR_FUND_AI"]) / df['QT_DOC_FUND_AI'].replace(0, np.nan)
df['aluno_professor_ed_basica_a_finais'] = (df['QT_MAT_FUND_AF'] / df["QT_TUR_FUND_AF"]) / df['QT_DOC_FUND_AF'].replace(0, np.nan)

# Recursos humanos
df['score_recursos_humanos'] = safe_sum('QT_PROF_PEDAGOGIA', 'QT_PROF_COORDENADOR', 'QT_PROF_PSICOLOGO')

# Materiais pedagógicos
df['score_pedagogico'] = safe_sum('IN_MATERIAL_PED_MULTIMIDIA', 'IN_MATERIAL_PED_CIENTIFICO', 'IN_MATERIAL_PED_JOGOS')

# Score geral de estrutura
df['score_estrutura'] = df['score_tecnologia'] + df['score_recursos_humanos'] + df['score_pedagogico']

print("Variáveis analíticas criadas")

# SALVAR CSV TRATAD0
output_file = "censo_tratado.csv"
df.to_csv(output_file, index=False)
print(f"Arquivo salvo como {output_file}")

files.download(output_file)
print("Processo concluido")

Realize o upload do arquivo do Censo Escolar (CSV)


Saving Censo_2023.csv to Censo_2023.csv
Arquivo CSV lido com sucesso
🔎 Shape inicial: (217625, 408)
filtro de escolas ativas: (178476, 408)
Valores divergentes tratados com sucesso
Variáveis analíticas criadas


/tmp/ipykernel_3862/2171724261.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['score_tecnologia'] = safe_sum('IN_INTERNET', 'IN_BANDA_LARGA', 'IN_INTERNET_ALUNOS')
/tmp/ipykernel_3862/2171724261.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['computadores_por_aluno'] = (safe_sum('QT_COMP_PORTATIL_ALUNO', 'QT_DESKTOP_ALUNO')) / qt_matriculas
/tmp/ipykernel_3862/2171724261.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

Arquivo salvo como censo_tratado.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Processo concluido


# Script de Limpeza

Este script tem como objetivo **preparar e enriquecer a base de dados do Censo Escolar 2023** para análises de Business Intelligence e Insights estratégicos sobre a educação básica no Brasil.

## Objetivos principais

1. **Limpeza e padronização dos dados**  
   - Substituição de valores especiais ou divergentes (como 88888 ou 9) por `NaN` para evitar distorções estatísticas.  
   - Filtragem apenas de **escolas ativas e com alunos matriculados**, garantindo que as análises reflitam a realidade atual do sistema escolar.

2. **Criação de métricas analíticas**  
   O script adiciona novas colunas que **transformam dados brutos em indicadores relevantes** para análise, por exemplo:  
   - **score_tecnologia**: soma de indicadores de acesso à internet e banda larga, permitindo avaliar a infraestrutura tecnológica das escolas.  
   - **computadores_por_aluno**: relação entre a quantidade de computadores (desktops + portáteis) e o número de alunos, fornecendo um indicador direto de recursos tecnológicos por estudante.  
   - **aluno_professor_ed_basica_a_iniciais/finais**: relação entre alunos e docentes em diferentes etapas da educação fundamental, permitindo análises de carga docente e atendimento educacional.  
   - **score_recursos_humanos** e **score_pedagogico**: indicadores que consolidam recursos humanos especializados e materiais pedagógicos disponíveis, oferecendo visão sobre a qualidade da estrutura educacional.  
   - **score_estrutura**: índice geral combinando tecnologia, recursos humanos e materiais pedagógicos, servindo como um indicador composto da qualidade estrutural da escola.

3. **Otimização de tipos de dados**  
   - Colunas categóricas (como localização e dependência administrativa) são convertidas para o tipo `category`, **reduzindo consumo de memória e acelerando operações futuras**.

## Por que essas métricas são importantes?

- Elas permitem **comparar escolas de diferentes regiões, portes e dependências administrativas** de forma padronizada.  
- Facilitam a **identificação de padrões e gargalos** na infraestrutura escolar, como baixa disponibilidade de computadores ou excesso de alunos por docente.  
- Servem como base para **visualizações e dashboards**, possibilitando análises exploratórias, relatórios de desempenho e estudos correlacionando infraestrutura com indicadores de aprendizagem.
